# Viral Genome Assembly and Variant Analysis

**Tier 3 — Applied Bioinformatics | Module 37 · Notebook 1**

*Prerequisites: Module 01 (NGS Fundamentals), Module 17 (Genome Assembly)*

---

**By the end of this notebook you will be able to:**
1. Describe unique challenges of viral sequencing: quasispecies, high mutation rate, small genomes
2. Perform reference-guided and de novo viral genome assembly
3. Call intra-host variants and minor alleles with LoFreq or iVar
4. Annotate viral variants using SnpEff with viral reference databases
5. Construct consensus genomes for phylogenetic analysis


**Key resources:**
- [iVar documentation](https://andersen-lab.github.io/ivar/html/)
- [LoFreq documentation](https://csb5.github.io/lofreq/)
- [NCBI Virus database](https://www.ncbi.nlm.nih.gov/labs/virus/vssi/)
- [Viral Bioinformatics Resource Center](https://virology.uvic.ca/)

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This workflow maps to production-style applied bioinformatics: pipeline design, model evaluation, and biological interpretation for decision-making.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Pipeline outputs are context-dependent: QC failures upstream can invalidate downstream interpretation.
- Batch effects and confounders can dominate signal if not controlled explicitly.
- Use uncertainty-aware decisions: rank candidates and keep rationale for every threshold.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

# CSV preview (DNA)
with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

# Variants preview
with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

# FASTA preview
print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

# Metadata preview
meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Module Overview](../README.md) | [Next: Phylodynamics →](./02_phylodynamics.ipynb)

---

## 1. Viral Sequencing Strategies

> Amplicon-based (ARTIC primers for SARS-CoV-2, HIV, dengue) vs metagenomic (unbiased). Oxford Nanopore rapid sequencing for outbreak response. Illumina deep sequencing for quasispecies. Typical genome sizes: RNA viruses 10–30 kb, DNA viruses 5 kb – 375 kb.

## 2. Reference-Guided Assembly

> Trim ARTIC primers with iVar trim. Align with BWA-MEM or Minimap2. Call variants. Generate consensus with iVar consensus or bcftools consensus. Coverage uniformity analysis.

In [ ]:
# Example: SARS-CoV-2 assembly with iVar
# !minimap2 -ax sr MN908947.3.fa sample_R1.fastq.gz sample_R2.fastq.gz | \
#     samtools sort -o sample.bam && samtools index sample.bam
# !ivar trim -i sample.bam -b nCoV-2019.primer.bed -p sample_trimmed -e
# !samtools mpileup -A -d 0 -Q 0 sample_trimmed.sorted.bam | ivar consensus -p consensus -n N

## 3. Intra-Host Variant Calling

> Minor variant calling at 1-5% frequency. LoFreq Poisson error model. Distinguish true variants from sequencing errors. Drug resistance mutations in HIV/HCV.

## 4. Viral Annotation

> VADR for automated viral genome annotation. SnpEff viral databases (flu, SARS-CoV-2 Pango lineage defining mutations). Nextclade for clade assignment.

## 5. Quality Assessment

> Coverage depth per position (acceptable > 20×). Genome completeness percentage. N content in consensus. Comparison with known reference strains.